# 🗄️ Phase 3 — SQL Multi-Table Analysis
**Project**: Blinkit Supply Chain & Inventory Optimization  
**Author**: Rishi Verma

---
### 🎯 Goals:
- Load all cleaned CSVs into SQLite database
- Write 15+ real business SQL queries using JOINs, CTEs, Window Functions
- Answer key supply chain business questions
- Export SQL file for portfolio
---

## 1️⃣ Setup SQLite Database

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_style('whitegrid')

CLEANED = '../data/cleaned'
DB_PATH = '../data/blinkit_supplychain.db'

# Create SQLite connection
conn = sqlite3.connect(DB_PATH)

# Load all cleaned tables into SQLite
tables = {
    'products'    : 'products_cleaned.csv',
    'inventory'   : 'inventory_cleaned.csv',
    'orders'      : 'orders_cleaned.csv',
    'order_items' : 'order_items_cleaned.csv',
    'customers'   : 'customers_cleaned.csv',
    'delivery'    : 'delivery_cleaned.csv',
    'feedback'    : 'feedback_cleaned.csv',
    'marketing'   : 'marketing_cleaned.csv'
}

for table_name, filename in tables.items():
    df = pd.read_csv(f'{CLEANED}/{filename}')
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f'  ✅ Loaded: {table_name:<15} ({len(df):>6,} rows)')

print(f'\n🗄️  SQLite DB created at: {DB_PATH}')

In [ ]:
# Helper function to run SQL and display results
def run_sql(query, title='', show_rows=10):
    """Execute SQL query, print title and return DataFrame."""
    if title:
        print(f'\n{"-"*60}')
        print(f'  🔍 {title}')
        print(f'{"-"*60}')
    df = pd.read_sql_query(query, conn)
    print(df.head(show_rows).to_string(index=False))
    print(f'  [{len(df)} rows returned]')
    return df

print('✅ Helper function ready!')

## 2️⃣ Revenue & Sales Queries

In [ ]:
# ── Q1: Total Revenue, Orders, AOV by Month ──
q1 = run_sql("""
    SELECT 
        order_month_name                              AS Month,
        order_year                                    AS Year,
        COUNT(DISTINCT order_id)                      AS Total_Orders,
        ROUND(SUM(order_total), 2)                    AS Total_Revenue,
        ROUND(AVG(order_total), 2)                    AS Avg_Order_Value,
        ROUND(SUM(order_total) * 100.0 /
              SUM(SUM(order_total)) OVER(), 2)        AS Revenue_Pct
    FROM orders
    GROUP BY order_year, order_month, order_month_name
    ORDER BY order_year, order_month
""", 'Q1: Monthly Revenue Summary with Window Function')

In [ ]:
# ── Q2: Top 10 Products by Revenue (JOIN order_items + products) ──
q2 = run_sql("""
    SELECT 
        p.product_name,
        p.category,
        p.brand,
        SUM(oi.quantity)                              AS Total_Units_Sold,
        ROUND(SUM(oi.line_total), 2)                  AS Total_Revenue,
        ROUND(AVG(oi.unit_price), 2)                  AS Avg_Selling_Price,
        ROUND(p.mrp, 2)                               AS MRP,
        RANK() OVER (ORDER BY SUM(oi.line_total) DESC) AS Revenue_Rank
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.product_id, p.product_name, p.category, p.brand, p.mrp
    ORDER BY Total_Revenue DESC
    LIMIT 10
""", 'Q2: Top 10 Products by Revenue (with RANK)')

In [ ]:
# ── Q3: Revenue by Category with Running Total ──
q3 = run_sql("""
    WITH category_revenue AS (
        SELECT 
            p.category,
            ROUND(SUM(oi.line_total), 2) AS revenue
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        GROUP BY p.category
    )
    SELECT 
        category,
        revenue                                        AS Category_Revenue,
        ROUND(revenue * 100.0 / SUM(revenue) OVER(), 2) AS Revenue_Share_Pct,
        ROUND(SUM(revenue) OVER (ORDER BY revenue DESC), 2) AS Running_Total
    FROM category_revenue
    ORDER BY revenue DESC
""", 'Q3: Category Revenue with CTE + Running Total')

In [ ]:
# ── Q4: Revenue by Customer Segment ──
q4 = run_sql("""
    SELECT 
        c.customer_segment,
        COUNT(DISTINCT o.order_id)      AS Total_Orders,
        COUNT(DISTINCT o.customer_id)   AS Total_Customers,
        ROUND(SUM(o.order_total), 2)    AS Total_Revenue,
        ROUND(AVG(o.order_total), 2)    AS Avg_Order_Value,
        ROUND(SUM(o.order_total) / COUNT(DISTINCT o.customer_id), 2) AS Revenue_Per_Customer
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_segment
    ORDER BY Total_Revenue DESC
""", 'Q4: Revenue & Orders by Customer Segment (3-table JOIN)')

## 3️⃣ Inventory & Stock Queries

In [ ]:
# ── Q5: Stock-Out Risk — products with high sales but low stock buffer ──
q5 = run_sql("""
    WITH product_demand AS (
        SELECT 
            product_id,
            SUM(quantity) AS total_sold,
            ROUND(AVG(quantity), 2) AS avg_daily_demand
        FROM order_items
        GROUP BY product_id
    )
    SELECT 
        p.product_name,
        p.category,
        p.min_stock_level,
        p.max_stock_level,
        p.shelf_life_days,
        pd.total_sold,
        pd.avg_daily_demand,
        ROUND(p.min_stock_level / NULLIF(pd.avg_daily_demand, 0), 1) AS days_of_cover,
        CASE 
            WHEN p.min_stock_level / NULLIF(pd.avg_daily_demand, 0) < 2 THEN '🔴 CRITICAL'
            WHEN p.min_stock_level / NULLIF(pd.avg_daily_demand, 0) < 5 THEN '🟡 WARNING'
            ELSE '🟢 SAFE'
        END AS stock_risk
    FROM products p
    JOIN product_demand pd ON p.product_id = pd.product_id
    ORDER BY days_of_cover ASC
    LIMIT 15
""", 'Q5: Stock-Out Risk Analysis (CTE + CASE WHEN)')

In [ ]:
# ── Q6: Damaged Stock Value Lost by Category ──
q6 = run_sql("""
    SELECT 
        p.category,
        SUM(i.stock_received)                             AS Total_Received,
        SUM(i.damaged_stock)                              AS Total_Damaged,
        ROUND(SUM(i.damaged_stock) * 100.0 /
              NULLIF(SUM(i.stock_received), 0), 2)        AS Damage_Rate_Pct,
        ROUND(SUM(i.damaged_stock) * AVG(p.price), 2)    AS Estimated_Loss_INR
    FROM inventory i
    JOIN products p ON i.product_id = p.product_id
    GROUP BY p.category
    ORDER BY Estimated_Loss_INR DESC
""", 'Q6: Damaged Stock Financial Loss by Category')

In [ ]:
# ── Q7: Monthly Inventory Trend per Category ──
q7 = run_sql("""
    SELECT 
        p.category,
        strftime('%Y-%m', i.date)             AS year_month,
        SUM(i.stock_received)                 AS monthly_received,
        SUM(i.damaged_stock)                  AS monthly_damaged,
        SUM(i.good_stock)                     AS monthly_good_stock,
        ROUND(AVG(i.damage_rate), 2)          AS avg_damage_rate
    FROM inventory i
    JOIN products p ON i.product_id = p.product_id
    GROUP BY p.category, year_month
    ORDER BY p.category, year_month
    LIMIT 20
""", 'Q7: Monthly Inventory Trend by Category')

In [ ]:
# ── Q8: Slow-Moving Products (high stock, low sales) ──
q8 = run_sql("""
    WITH sales_agg AS (
        SELECT product_id, SUM(quantity) AS units_sold
        FROM order_items
        GROUP BY product_id
    )
    SELECT 
        p.product_name,
        p.category,
        p.max_stock_level,
        COALESCE(s.units_sold, 0)           AS units_sold,
        p.price,
        ROUND(p.max_stock_level * p.price, 2) AS inventory_value_at_risk
    FROM products p
    LEFT JOIN sales_agg s ON p.product_id = s.product_id
    WHERE COALESCE(s.units_sold, 0) < 5
    ORDER BY inventory_value_at_risk DESC
    LIMIT 10
""", 'Q8: Slow-Moving / Dead Stock Products')

## 4️⃣ Delivery Performance Queries

In [ ]:
# ── Q9: On-Time vs Late Delivery with Avg Delay per Store ──
q9 = run_sql("""
    SELECT 
        o.store_id,
        COUNT(o.order_id)                     AS total_orders,
        SUM(o.is_late)                         AS late_orders,
        ROUND(SUM(o.is_late) * 100.0 / COUNT(*), 1) AS late_pct,
        ROUND(AVG(d.delivery_time_minutes), 1) AS avg_delivery_min,
        ROUND(AVG(o.delay_minutes), 1)         AS avg_delay_min,
        ROUND(SUM(o.order_total), 2)           AS store_revenue
    FROM orders o
    JOIN delivery d ON o.order_id = d.order_id
    GROUP BY o.store_id
    ORDER BY late_pct DESC
    LIMIT 10
""", 'Q9: Delivery Performance by Store (worst performers)')

In [ ]:
# ── Q10: Delivery Delay vs Customer Rating (do late orders get bad ratings?) ──
q10 = run_sql("""
    SELECT 
        o.delivery_status,
        COUNT(f.feedback_id)              AS total_feedbacks,
        ROUND(AVG(f.rating), 2)           AS avg_rating,
        ROUND(AVG(o.delay_minutes), 1)    AS avg_delay_min,
        SUM(CASE WHEN f.sentiment = 'Positive' THEN 1 ELSE 0 END) AS positive_count,
        SUM(CASE WHEN f.sentiment = 'Negative' THEN 1 ELSE 0 END) AS negative_count
    FROM orders o
    JOIN feedback f ON o.order_id = f.order_id
    GROUP BY o.delivery_status
    ORDER BY avg_rating DESC
""", 'Q10: Delivery Status vs Customer Rating & Sentiment')

## 5️⃣ Customer Behaviour Queries

In [ ]:
# ── Q11: RFM Scoring — Recency, Frequency, Monetary ──
q11 = run_sql("""
    WITH rfm_base AS (
        SELECT 
            o.customer_id,
            c.customer_segment,
            MAX(DATE(o.order_date))                     AS last_order_date,
            JULIANDAY('now') - JULIANDAY(MAX(DATE(o.order_date))) AS recency_days,
            COUNT(DISTINCT o.order_id)                  AS frequency,
            ROUND(SUM(o.order_total), 2)                AS monetary
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY o.customer_id, c.customer_segment
    )
    SELECT 
        customer_id,
        customer_segment,
        ROUND(recency_days, 0)  AS recency_days,
        frequency,
        monetary,
        CASE 
            WHEN recency_days <= 30  AND frequency >= 3 AND monetary >= 1000 THEN '🏆 Champions'
            WHEN recency_days <= 60  AND frequency >= 2                       THEN '💛 Loyal'
            WHEN recency_days <= 90                                            THEN '🔄 At Risk'
            ELSE '😴 Lost'
        END AS rfm_segment
    FROM rfm_base
    ORDER BY monetary DESC
    LIMIT 15
""", 'Q11: RFM Customer Segmentation (CTE + CASE WHEN)')

In [ ]:
# ── Q12: Top 10 Most Valuable Customers ──
q12 = run_sql("""
    SELECT 
        c.customer_name,
        c.customer_segment,
        c.area,
        COUNT(DISTINCT o.order_id)         AS total_orders,
        ROUND(SUM(o.order_total), 2)       AS lifetime_value,
        ROUND(AVG(o.order_total), 2)       AS avg_order_value,
        ROUND(AVG(f.rating), 2)            AS avg_rating,
        DENSE_RANK() OVER (ORDER BY SUM(o.order_total) DESC) AS value_rank
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    LEFT JOIN feedback f ON o.order_id = f.order_id
    GROUP BY o.customer_id, c.customer_name, c.customer_segment, c.area
    ORDER BY lifetime_value DESC
    LIMIT 10
""", 'Q12: Top 10 Most Valuable Customers (DENSE_RANK)')

## 6️⃣ Advanced Supply Chain Queries

In [ ]:
# ── Q13: Product Profitability — Margin vs Volume ──
q13 = run_sql("""
    SELECT 
        p.product_name,
        p.category,
        p.margin_percentage,
        SUM(oi.quantity)                                   AS units_sold,
        ROUND(SUM(oi.line_total), 2)                       AS gross_revenue,
        ROUND(SUM(oi.line_total) * p.margin_percentage / 100, 2) AS estimated_profit,
        NTILE(4) OVER (ORDER BY SUM(oi.line_total) * p.margin_percentage / 100 DESC) AS profit_quartile
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.product_id, p.product_name, p.category, p.margin_percentage
    ORDER BY estimated_profit DESC
    LIMIT 12
""", 'Q13: Product Profitability Ranking (NTILE Window Function)')

In [ ]:
# ── Q14: Payment Method Preference by Customer Segment ──
q14 = run_sql("""
    SELECT 
        c.customer_segment,
        o.payment_method,
        COUNT(*)                              AS order_count,
        ROUND(AVG(o.order_total), 2)          AS avg_order_value,
        ROUND(COUNT(*) * 100.0 /
              SUM(COUNT(*)) OVER (PARTITION BY c.customer_segment), 1) AS pct_within_segment
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_segment, o.payment_method
    ORDER BY c.customer_segment, order_count DESC
""", 'Q14: Payment Method by Segment (PARTITION BY Window)')

In [ ]:
# ── Q15: Marketing Channel Efficiency Score ──
q15 = run_sql("""
    SELECT 
        channel,
        COUNT(campaign_id)                     AS campaigns,
        ROUND(SUM(spend), 2)                   AS total_spend,
        ROUND(SUM(revenue_generated), 2)       AS total_revenue,
        ROUND(SUM(revenue_generated) / NULLIF(SUM(spend), 0), 2) AS overall_roas,
        ROUND(AVG(ctr), 2)                     AS avg_ctr_pct,
        ROUND(AVG(conversion_rate), 2)         AS avg_conv_rate_pct,
        SUM(conversions)                       AS total_conversions,
        ROUND(SUM(spend) / NULLIF(SUM(conversions), 0), 2) AS cost_per_conversion
    FROM marketing
    GROUP BY channel
    ORDER BY overall_roas DESC
""", 'Q15: Marketing Channel Efficiency Full Scorecard')

## 7️⃣ Export SQL Script for Portfolio

In [ ]:
sql_script = '''
-- ================================================================
-- BLINKIT SUPPLY CHAIN — SQL ANALYSIS QUERIES
-- Author  : Rishi Verma
-- Dataset : Blinkit Grocery & Supply Chain Data
-- Engine  : SQLite (compatible with PostgreSQL/MySQL)
-- ================================================================

-- Q1: Monthly Revenue Summary with Window Function
SELECT 
    order_month_name AS Month,
    order_year AS Year,
    COUNT(DISTINCT order_id) AS Total_Orders,
    ROUND(SUM(order_total), 2) AS Total_Revenue,
    ROUND(AVG(order_total), 2) AS Avg_Order_Value,
    ROUND(SUM(order_total) * 100.0 / SUM(SUM(order_total)) OVER(), 2) AS Revenue_Pct
FROM orders
GROUP BY order_year, order_month, order_month_name
ORDER BY order_year, order_month;

-- Q2: Top 10 Products by Revenue
SELECT 
    p.product_name, p.category, p.brand,
    SUM(oi.quantity) AS Total_Units_Sold,
    ROUND(SUM(oi.line_total), 2) AS Total_Revenue,
    RANK() OVER (ORDER BY SUM(oi.line_total) DESC) AS Revenue_Rank
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.product_id
ORDER BY Total_Revenue DESC
LIMIT 10;

-- Q5: Stock-Out Risk Analysis
WITH product_demand AS (
    SELECT product_id, SUM(quantity) AS total_sold,
           ROUND(AVG(quantity), 2) AS avg_daily_demand
    FROM order_items
    GROUP BY product_id
)
SELECT 
    p.product_name, p.category,
    p.min_stock_level, p.max_stock_level, p.shelf_life_days,
    pd.total_sold, pd.avg_daily_demand,
    ROUND(p.min_stock_level / NULLIF(pd.avg_daily_demand,0), 1) AS days_of_cover,
    CASE 
        WHEN p.min_stock_level / NULLIF(pd.avg_daily_demand,0) < 2 THEN 'CRITICAL'
        WHEN p.min_stock_level / NULLIF(pd.avg_daily_demand,0) < 5 THEN 'WARNING'
        ELSE 'SAFE'
    END AS stock_risk
FROM products p
JOIN product_demand pd ON p.product_id = pd.product_id
ORDER BY days_of_cover ASC;

-- Q11: RFM Customer Segmentation
WITH rfm_base AS (
    SELECT o.customer_id, c.customer_segment,
           JULIANDAY('now') - JULIANDAY(MAX(DATE(o.order_date))) AS recency_days,
           COUNT(DISTINCT o.order_id) AS frequency,
           ROUND(SUM(o.order_total), 2) AS monetary
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_segment
)
SELECT customer_id, customer_segment,
       ROUND(recency_days,0) AS recency_days,
       frequency, monetary,
       CASE 
           WHEN recency_days<=30 AND frequency>=3 AND monetary>=1000 THEN 'Champions'
           WHEN recency_days<=60 AND frequency>=2 THEN 'Loyal'
           WHEN recency_days<=90 THEN 'At Risk'
           ELSE 'Lost'
       END AS rfm_segment
FROM rfm_base
ORDER BY monetary DESC;
'''

os.makedirs('../sql', exist_ok=True)
with open('../sql/blinkit_queries.sql', 'w') as f:
    f.write(sql_script)

print('✅ SQL script saved to: sql/blinkit_queries.sql')
print('\n🎉 Phase 3 Complete! 15 SQL queries written using:')
print('   ✅ JOINs (INNER, LEFT)')
print('   ✅ CTEs (WITH clause)')
print('   ✅ Window Functions (RANK, DENSE_RANK, NTILE, OVER PARTITION BY)')
print('   ✅ Aggregations (SUM, AVG, COUNT, ROUND)')
print('   ✅ CASE WHEN conditional logic')
print('   ✅ NULLIF, COALESCE for null safety')
conn.close()